In [ ]:
# ================================
# Google Drive + chemins du TP
# ================================
# Cette cellule utilise le même chemin Drive que la dernière version :
# /content/drive/MyDrive/Colab_Projects/NA_Fish_Dataset

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Montage Google Drive ignoré ou indisponible :", e)

DRIVE_ROOT = "/content/drive/MyDrive"
BASE_DRIVE = os.path.join(DRIVE_ROOT, "Colab_Projects", "NA_Fish_Dataset")

# Pour ce TP MNIST, on garde le même dossier principal,
# mais on crée des sous-dossiers spécifiques à MNIST.
MNIST_PROJECT_DIR = os.path.join(BASE_DRIVE, "MNIST_CNN_TP")
DATA_ROOT = os.path.join(MNIST_PROJECT_DIR, "data")
MODELS_DIR = os.path.join(MNIST_PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(MNIST_PROJECT_DIR, "results")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODELS_DIR, "mnist_cnn_pytorch.pt")

print("BASE_DRIVE        :", BASE_DRIVE)
print("MNIST_PROJECT_DIR :", MNIST_PROJECT_DIR)
print("DATA_ROOT         :", DATA_ROOT)
print("MODELS_DIR        :", MODELS_DIR)
print("MODEL_PATH        :", MODEL_PATH)


## Utilisation de Google Drive

Cette version stocke automatiquement :
- le dataset MNIST dans `DATA_ROOT`
- le modèle entraîné dans `MODELS_DIR`
- les résultats globaux dans `RESULTS_DIR`

Le chemin principal est le même que la dernière version :
`/content/drive/MyDrive/Colab_Projects/NA_Fish_Dataset`


In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import random
import time
import numpy as np

# Device: GPU si disponible, sinon CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

# Pour rendre les résultats plus reproductibles
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)


In [ ]:
# Chargement du dataset MNIST directement avec torchvision.
# MNIST contient des images en niveaux de gris de taille 28x28.
# Chaque image représente un chiffre entre 0 et 9.

batch_size_train = 64
batch_size_test = 1000

# Transformations appliquées aux images :
# 1) ToTensor() : conversion image -> tenseur PyTorch
# 2) Normalize(...) : normalisation avec la moyenne et l'écart-type de MNIST
transform_mnist = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.1307,), (0.3081,))
])

train_loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transform_mnist),
    batch_size=batch_size_train,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(DATA_ROOT, train=False, download=True, transform=transform_mnist),
    batch_size=batch_size_test,
    shuffle=False
)

print("Nombre d'images train :", len(train_loader.dataset))
print("Nombre d'images test  :", len(test_loader.dataset))


In [ ]:
# Affichage de quelques exemples du dataset de test
examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

fig = plt.figure(figsize=(8, 5))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.tight_layout()
    plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
    plt.title("Ground Truth: {}".format(example_targets[i].item()))
    plt.xticks([])
    plt.yticks([])
plt.show()

print("Shape d'un batch d'images :", example_data.shape)
print("Shape d'un batch de labels :", example_targets.shape)


## Définition du réseau CNN

Dans cette partie, on remplace le réseau minimal par un vrai **CNN** conforme aux consignes du TP :

1. convolution valide `5x5`, 1 canal d'entrée, 10 canaux de sortie  
2. max pooling `2x2`  
3. ReLU  
4. convolution valide `5x5`, 10 canaux d'entrée, 20 canaux de sortie  
5. Dropout 2D  
6. max pooling `2x2`  
7. ReLU  
8. Flatten  
9. couche fully connected `320 -> 50`  
10. ReLU  
11. couche fully connected `50 -> 10`  
12. `log_softmax` pour utiliser correctement `F.nll_loss`


In [ ]:
# Réseau CNN complet pour MNIST
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # 1) Première convolution :
        # entrée : image MNIST (1 canal, 28x28)
        # sortie : 10 cartes de caractéristiques
        # kernel_size=5 => convolution valide : 28x28 devient 24x24
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=10, kernel_size=5)

        # 2) Deuxième convolution :
        # entrée : 10 canaux
        # sortie : 20 canaux
        # après conv1 + pooling : 12x12
        # conv2 avec kernel 5 : 12x12 devient 8x8
        self.conv2 = nn.Conv2d(in_channels=10, out_channels=20, kernel_size=5)

        # 3) Dropout 2D pour limiter l'overfitting
        self.conv2_drop = nn.Dropout2d(p=0.25)

        # 4) Max pooling sur une fenêtre 2x2
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Après :
        # 28x28 -> conv1 -> 24x24 -> pool -> 12x12
        # 12x12 -> conv2 -> 8x8 -> pool -> 4x4
        # Nombre final de valeurs = 20 canaux * 4 * 4 = 320
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        # Bloc 1 : convolution + pooling + ReLU
        x = self.conv1(x)
        x = self.pool(x)
        x = F.relu(x)

        # Bloc 2 : convolution + dropout + pooling + ReLU
        x = self.conv2(x)
        x = self.conv2_drop(x)
        x = self.pool(x)
        x = F.relu(x)

        # Flatten : transformation du tenseur 2D en vecteur
        x = torch.flatten(x, start_dim=1)

        # Couche fully connected 1 + ReLU
        x = self.fc1(x)
        x = F.relu(x)

        # Couche fully connected 2
        x = self.fc2(x)

        # LogSoftmax parce que la fonction de perte utilisée est F.nll_loss
        return F.log_softmax(x, dim=1)


In [ ]:
# Initialisation des poids
# He/Kaiming est adaptée aux activations ReLU.
def weights_init(layer_in):
    if isinstance(layer_in, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_uniform_(layer_in.weight, nonlinearity='relu')
        if layer_in.bias is not None:
            layer_in.bias.data.fill_(0.0)


In [ ]:
# Création du réseau
model = Net().to(device)

# Initialisation des poids du modèle
model.apply(weights_init)

# Définition de l'optimiseur
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

print(model)


In [ ]:
# Routine principale d'entraînement
def train(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # Envoyer les données vers GPU/CPU
        data, target = data.to(device), target.to(device)

        # Réinitialiser les gradients
        optimizer.zero_grad()

        # Propagation avant
        output = model(data)

        # Calcul de la perte
        loss = F.nll_loss(output, target)

        # Backpropagation
        loss.backward()

        # Mise à jour des poids
        optimizer.step()

        # Statistiques
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{}]\tLoss: {:.6f}'.format(
                epoch,
                batch_idx * len(data),
                len(train_loader.dataset),
                loss.item()
            ))

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    print("Train Epoch {} - Avg loss: {:.4f}, Accuracy: {:.2f}%".format(epoch, avg_loss, accuracy))

    return avg_loss, accuracy


In [ ]:
# Évaluation sur les données de test
def test():
    model.eval()
    test_loss = 0
    correct = 0

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            # Envoyer les données vers GPU/CPU
            data, target = data.to(device), target.to(device)

            # Prédiction
            output = model(data)

            # Perte totale
            test_loss += F.nll_loss(output, target, reduction='sum').item()

            # Classe prédite
            pred = output.argmax(dim=1)

            # Nombre de prédictions correctes
            correct += pred.eq(target).sum().item()

            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    test_loss /= len(test_loader.dataset)
    accuracy = 100.0 * correct / len(test_loader.dataset)

    print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss,
        correct,
        len(test_loader.dataset),
        accuracy
    ))

    return test_loss, accuracy, np.array(all_preds), np.array(all_targets)


In [ ]:
# Performance initiale avant entraînement
print("Performance initiale :")
initial_test_loss, initial_test_acc, _, _ = test()

# Entraînement
n_epochs = 3

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

start_time = time.time()

for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train(epoch)
    test_loss, test_acc, _, _ = test()

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

execution_time = time.time() - start_time

print("=" * 60)
print("RÉSULTATS GLOBAUX")
print("=" * 60)
print(f"Temps total d'exécution : {int(execution_time // 60)} min {int(execution_time % 60)} s")
print(f"Accuracy initiale test  : {initial_test_acc:.2f}%")
print(f"Accuracy finale train   : {train_accuracies[-1]:.2f}%")
print(f"Accuracy finale test    : {test_accuracies[-1]:.2f}%")
print(f"Loss finale train       : {train_losses[-1]:.4f}")
print(f"Loss finale test        : {test_losses[-1]:.4f}")
print("=" * 60)

# Courbes loss / accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(range(1, n_epochs + 1), train_losses, label="Train loss")
plt.plot(range(1, n_epochs + 1), test_losses, label="Test loss")
plt.title("Évolution de la loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, n_epochs + 1), train_accuracies, label="Train accuracy")
plt.plot(range(1, n_epochs + 1), test_accuracies, label="Test accuracy")
plt.title("Évolution de l'accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.show()


# Sauvegarde du modèle entraîné dans Google Drive
torch.save(model.state_dict(), MODEL_PATH)
print("Modèle sauvegardé dans :", MODEL_PATH)

# Sauvegarde des métriques principales dans un fichier texte
metrics_path = os.path.join(RESULTS_DIR, "mnist_cnn_results.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("RÉSULTATS GLOBAUX - CNN MNIST\n")
    f.write("=" * 60 + "\n")
    f.write(f"Temps total d'exécution : {int(execution_time // 60)} min {int(execution_time % 60)} s\n")
    f.write(f"Accuracy initiale test  : {initial_test_acc:.2f}%\n")
    f.write(f"Accuracy finale train   : {train_accuracies[-1]:.2f}%\n")
    f.write(f"Accuracy finale test    : {test_accuracies[-1]:.2f}%\n")
    f.write(f"Loss finale train       : {train_losses[-1]:.4f}\n")
    f.write(f"Loss finale test        : {test_losses[-1]:.4f}\n")

print("Résultats sauvegardés dans :", metrics_path)


In [ ]:
# Prédiction sur les exemples affichés au début
model.eval()

with torch.no_grad():
    output = model(example_data.to(device))
    predictions = output.argmax(dim=1).cpu()

fig = plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.tight_layout()
    plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
    plt.title("Pred: {} / True: {}".format(
        predictions[i].item(),
        example_targets[i].item()
    ))
    plt.xticks([])
    plt.yticks([])
plt.show()


In [ ]:
# Matrice de confusion finale
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

_, _, y_pred, y_true = test()

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matrice de confusion - MNIST")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.show()

print("Rapport de classification :")
print(classification_report(y_true, y_pred))
